In [1]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!pip -q install ollama pymupdf

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%                           17.0%                                              30.8%######################                           66.3%#########################################       93.5%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [2]:
import subprocess, time, glob, json, re, hashlib, datetime, unicodedata
from difflib import SequenceMatcher
from pathlib import Path
import pandas as pd

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

MODEL = "qwen2.5vl:7b"
PDFS = {re.search(r"bilan_group_(\d+)", p).group(1): p for p in glob.glob("/kaggle/input/**/bilan_group_*.pdf", recursive=True)}
GOLD = {re.search(r"bilan_(\d+)\.json", p).group(1): json.load(open(p, encoding="utf-8"))
        for p in glob.glob("/kaggle/input/**/bilan_[0-9]*.json", recursive=True)}
RESULTS = Path("/kaggle/working/results")
RESULTS.mkdir(exist_ok=True)
print("bilans:", sorted(PDFS, key=int), "| references:", sorted(GOLD, key=int))

!ollama pull {MODEL}

bilans: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10'] | references: ['3', '6', '8', '10']
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling a99b7f834d75:   0% ▕                  ▏ 117 KB/6.0 GB                  pulling manifest 
pulling a99b7f834d75:   0% ▕                  ▏  11 MB/6.0 GB                  pulling manifest 
pulling a99b7f834d75:   2% ▕                  ▏  99 MB/6.0 GB                  pulling manifest 
pulling a99b7f834d75:   3% ▕                  ▏ 181 MB/6.0 GB                  pulling manifest 
pulling a99b7f834d75:   4% ▕                  ▏ 231 MB/6.0 GB                  pulling manifest 
pulling a99b7f834d75:   5% ▕                  ▏ 326 MB/6.0 GB                  pulling manifest 
pulling a99b7f834d75:   7% ▕█                 ▏ 425 MB/6.0 GB                  pulling 

In [3]:
import fitz
from PIL import Image

PAGES = {
    "6":  ["ACTIF", "PASSIF"],
    "3":  ["ACTIF", "PASSIF"],
    "8":  ["ACTIF", "ACTIF", "PASSIF"],
    "10": ["ACTIF", "ACTIF", "PASSIF"],
    "2":  ["ACTIF", "PASSIF"],
    "4":  ["ACTIF", "PASSIF"],
}

def images(setup, bilan, page):
    pix = fitz.open(PDFS[bilan])[page].get_pixmap(dpi=300, colorspace=fitz.csGRAY)
    img = Image.frombytes("L", (pix.width, pix.height), pix.samples)
    img.thumbnail((1800, 1800))
    parts = [img]
    if "strips" in setup:
        w, h = img.size
        header = img.crop((0, 0, w, int(h * 0.18)))
        rest = img.crop((0, int(h * 0.45), w, h))
        bottom = Image.new("L", (w, header.height + rest.height), 255)
        bottom.paste(header, (0, 0))
        bottom.paste(rest, (0, header.height))
        parts = [img.crop((0, 0, w, int(h * 0.60))), bottom]
    paths = []
    for i, part in enumerate(parts):
        path = f"/kaggle/working/{bilan}_{page}_{i}.png"
        part.save(path)
        paths.append(path)
    return paths

/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


In [4]:
CGNC_LINES = [
    ("AA",   "ACTIF", "A", "IMMOBILISATION EN NON VALEURS", "subtotal"),
    ("AA1",  "ACTIF", "A", "Frais préliminaires", "line"),
    ("AA2",  "ACTIF", "A", "Charges à répartir sur plusieurs exercices", "line"),
    ("AA3",  "ACTIF", "A", "Primes de remboursement des obligations", "line"),
    ("AB",   "ACTIF", "B", "IMMOBILISATIONS INCORPORELLES", "subtotal"),
    ("AB1",  "ACTIF", "B", "Immobilisation en recherche et développement", "line"),
    ("AB2",  "ACTIF", "B", "Brevets, marques, droits et valeurs assimilés", "line"),
    ("AB3",  "ACTIF", "B", "Fonds commercial", "line"),
    ("AB4",  "ACTIF", "B", "Autres immobilisations incorporelles", "line"),
    ("AC",   "ACTIF", "C", "IMMOBILISATIONS CORPORELLES", "subtotal"),
    ("AC1",  "ACTIF", "C", "Terrains", "line"),
    ("AC2",  "ACTIF", "C", "Constructions", "line"),
    ("AC3",  "ACTIF", "C", "Installations techniques, matériel et outillage", "line"),
    ("AC4",  "ACTIF", "C", "Matériel de transport", "line"),
    ("AC5",  "ACTIF", "C", "Mobilier, matériel de bureau et aménagements divers", "line"),
    ("AC6",  "ACTIF", "C", "Autres immobilisations corporelles", "line"),
    ("AC7",  "ACTIF", "C", "Immobilisations corporelles en cours", "line"),
    ("AD",   "ACTIF", "D", "IMMOBILISATIONS FINANCIERES", "subtotal"),
    ("AD1",  "ACTIF", "D", "Prêts immobilisés", "line"),
    ("AD2",  "ACTIF", "D", "Autres créances financières", "line"),
    ("AD3",  "ACTIF", "D", "Titres de participation", "line"),
    ("AD4",  "ACTIF", "D", "Autres titres immobilisés", "line"),
    ("AE",   "ACTIF", "E", "ECARTS DE CONVERSION - ACTIF", "subtotal"),
    ("AE1",  "ACTIF", "E", "Diminution des créances immobilisées", "line"),
    ("AE2",  "ACTIF", "E", "Augmentation des dettes de financement", "line"),
    ("AT1",  "ACTIF", "TOTAL", "TOTAL I (A+B+C+D+E)", "total"),
    ("AF",   "ACTIF", "F", "STOCKS", "subtotal"),
    ("AF1",  "ACTIF", "F", "Marchandises", "line"),
    ("AF2",  "ACTIF", "F", "Matières et fournitures consommables", "line"),
    ("AF3",  "ACTIF", "F", "Produits en cours", "line"),
    ("AF4",  "ACTIF", "F", "Produits intermédiaires et produits résiduels", "line"),
    ("AF5",  "ACTIF", "F", "Produits finis", "line"),
    ("AG",   "ACTIF", "G", "CREANCES DE L'ACTIF CIRCULANT", "subtotal"),
    ("AG1",  "ACTIF", "G", "Fournisseurs débiteurs, avances et acomptes", "line"),
    ("AG2",  "ACTIF", "G", "Clients et comptes rattachés", "line"),
    ("AG3",  "ACTIF", "G", "Personnel", "line"),
    ("AG4",  "ACTIF", "G", "Etat", "line"),
    ("AG5",  "ACTIF", "G", "Comptes d'associés", "line"),
    ("AG6",  "ACTIF", "G", "Autres débiteurs", "line"),
    ("AG7",  "ACTIF", "G", "Comptes de régularisation - Actif", "line"),
    ("AH",   "ACTIF", "H", "TITRES ET VALEURS DE PLACEMENT", "subtotal"),
    ("AI",   "ACTIF", "I", "ECARTS DE CONVERSION - ACTIF (éléments circulants)", "subtotal"),
    ("AT2",  "ACTIF", "TOTAL", "TOTAL II (F+G+H+I)", "total"),
    ("ATR",  "ACTIF", "TR", "TRESORERIE - ACTIF", "subtotal"),
    ("ATR1", "ACTIF", "TR", "Chèques et valeurs à encaisser", "line"),
    ("ATR2", "ACTIF", "TR", "Banques, T.G. et C.C.P.", "line"),
    ("ATR3", "ACTIF", "TR", "Caisses, régies et accréditifs", "line"),
    ("AT3",  "ACTIF", "TOTAL", "TOTAL III", "total"),
    ("ATG",  "ACTIF", "TOTAL", "TOTAL GENERAL (I+II+III)", "total"),

    ("PA0",  "PASSIF", "A", "CAPITAUX PROPRES", "header"),
    ("PA1",  "PASSIF", "A", "Capital social ou personnel", "line"),
    ("PA2",  "PASSIF", "A", "Moins : Actionnaires, capital souscrit non appelé", "line"),
    ("PA3",  "PASSIF", "A", "Capital appelé, dont versé", "line"),
    ("PA4",  "PASSIF", "A", "Primes d'émission, de fusion, d'apport", "line"),
    ("PA5",  "PASSIF", "A", "Ecarts de réévaluation", "line"),
    ("PA6",  "PASSIF", "A", "Réserve légale", "line"),
    ("PA7",  "PASSIF", "A", "Autres réserves", "line"),
    ("PA8",  "PASSIF", "A", "Report à nouveau", "line"),
    ("PA9",  "PASSIF", "A", "Résultats nets en instance d'affectation", "line"),
    ("PA10", "PASSIF", "A", "Résultat net de l'exercice", "line"),
    ("PA",   "PASSIF", "A", "Total des capitaux propres (A)", "subtotal"),
    ("PB",   "PASSIF", "B", "CAPITAUX PROPRES ASSIMILES (B)", "subtotal"),
    ("PB1",  "PASSIF", "B", "Subventions d'investissement", "line"),
    ("PB2",  "PASSIF", "B", "Provisions réglementées", "line"),
    ("PC",   "PASSIF", "C", "DETTES DE FINANCEMENT (C)", "subtotal"),
    ("PC1",  "PASSIF", "C", "Emprunts obligataires", "line"),
    ("PC2",  "PASSIF", "C", "Autres dettes de financement", "line"),
    ("PD",   "PASSIF", "D", "PROVISIONS DURABLES POUR RISQUES ET CHARGES (D)", "subtotal"),
    ("PD1",  "PASSIF", "D", "Provisions pour risques", "line"),
    ("PD2",  "PASSIF", "D", "Provisions pour charges", "line"),
    ("PE",   "PASSIF", "E", "COMPTE DE LIAISON (E)", "subtotal"),
    ("PF",   "PASSIF", "F", "ECARTS DE CONVERSION - PASSIF (F)", "subtotal"),
    ("PF1",  "PASSIF", "F", "Augmentation des créances immobilisées", "line"),
    ("PF2",  "PASSIF", "F", "Diminution des dettes de financement", "line"),
    ("PT1",  "PASSIF", "TOTAL", "TOTAL I (A+B+C+D+E+F)", "total"),
    ("PG",   "PASSIF", "G", "DETTES DU PASSIF CIRCULANT (G)", "subtotal"),
    ("PG1",  "PASSIF", "G", "Fournisseurs et comptes rattachés", "line"),
    ("PG2",  "PASSIF", "G", "Clients créditeurs, avances et acomptes", "line"),
    ("PG3",  "PASSIF", "G", "Personnel", "line"),
    ("PG4",  "PASSIF", "G", "Organismes sociaux", "line"),
    ("PG5",  "PASSIF", "G", "Etat", "line"),
    ("PG6",  "PASSIF", "G", "Comptes d'associés", "line"),
    ("PG7",  "PASSIF", "G", "Autres créanciers", "line"),
    ("PG8",  "PASSIF", "G", "Comptes de régularisation - Passif", "line"),
    ("PH",   "PASSIF", "H", "AUTRES PROVISIONS POUR RISQUES ET CHARGES (H)", "subtotal"),
    ("PI",   "PASSIF", "I", "ECARTS DE CONVERSION - PASSIF (I)", "subtotal"),
    ("PT2",  "PASSIF", "TOTAL", "TOTAL II (G+H+I)", "total"),
    ("PTR",  "PASSIF", "TR", "TRESORERIE - PASSIF", "subtotal"),
    ("PTR1", "PASSIF", "TR", "Crédits d'escompte", "line"),
    ("PTR2", "PASSIF", "TR", "Crédits de trésorerie", "line"),
    ("PTR3", "PASSIF", "TR", "Banques de régularisation", "line"),
    ("PT3",  "PASSIF", "TOTAL", "TOTAL III", "total"),
    ("PTG",  "PASSIF", "TOTAL", "TOTAL GENERAL (I+II+III)", "total"),
]

LABELS = {code: (side, section, label, kind) for code, side, section, label, kind in CGNC_LINES}
print(sum(s == "ACTIF" for s, *_ in LABELS.values()), "ACTIF lines,", sum(s == "PASSIF" for s, *_ in LABELS.values()), "PASSIF lines")

SIGN = {"PA2": -1, "PA3": 0}
TOTALS = {"AT1": "ABCDE", "AT2": "FGHI", "AT3": ["TR"], "PT1": "ABCDEF", "PT2": "GHI", "PT3": ["TR"]}
COLUMNS = {"ACTIF": ["brut", "amort", "n", "n1"], "PASSIF": ["n", "n1"]}
CODES = {side: [c for c, (s, *_) in LABELS.items() if s == side] for side in COLUMNS}

49 ACTIF lines, 44 PASSIF lines


In [6]:
NUMBER = {"anyOf": [{"type": "number"}, {"type": "null"}]}

def obj(fields):
    return {"type": "object", "properties": fields, "required": list(fields), "additionalProperties": False}

def rows_schema(side):
    k = len(COLUMNS[side])
    row = obj({"label": {"type": "string"}, "values": {"type": "array", "items": NUMBER, "minItems": k, "maxItems": k}})
    return obj({"lignes": {"type": "array", "items": row}})

def codes_schema(side):
    return obj({code: obj({"n": NUMBER, "n1": NUMBER}) for code in CODES[side]})

ROLE = """You are a Moroccan chartered accountant (CGNC chart of accounts).
You get one page of a balance sheet (bilan). Read the table and answer in JSON."""

NUMBERS = """- Write amounts as numbers: "1 234 567,89" becomes 1234567.89. Keep the minus sign.
- Empty cell: null. Never invent an amount.
- Answer with the JSON only."""

COLUMN_NAMES = {
    "ACTIF": "4 amount columns, left to right: Brut, Amortissements et provisions, Net (current year), Net (previous year)",
    "PASSIF": "2 amount columns, left to right: Exercice (current year), Exercice précédent (previous year)",
}

ROW_EXAMPLE = {
    "ACTIF": '{"lignes": [{"label": "Frais préliminaires", "values": [2700.0, 2160.0, 540.0, 1080.0]}, {"label": "Primes de remboursement des obligations", "values": [null, null, null, null]}]}',
    "PASSIF": '{"lignes": [{"label": "Réserve légale", "values": [275096.17, 275096.17]}, {"label": "Autres réserves", "values": [null, null]}]}',
}

def system_prompt(setup, side):
    if "codes" in setup:
        current = "Net amount of the current year" if side == "ACTIF" else "Exercice amount (current year)"
        previous = "Net amount of the previous year" if side == "ACTIF" else "Exercice précédent amount (previous year)"
        lines = "\n".join(f"{code}: {LABELS[code][2]}" for code in CODES[side])
        task = f"""Return every code below. For each code, read its row on the page:
- n = the {current}, n1 = the {previous}.
- If the row is empty or not on this page: null.
{NUMBERS}

Codes:
{lines}"""
    else:
        task = f"""Copy the table exactly, row by row, from top to bottom. Do not compute, sum or skip anything.
- One entry per printed row, even if the row has no amounts. Total rows too.
- label: the row text exactly as printed (in French).
- values: the {COLUMN_NAMES[side]}.
{NUMBERS}

Example:
{ROW_EXAMPLE[side]}"""
    return ROLE + "\n\n" + task

print(system_prompt("transcribe", "ACTIF"))

You are a Moroccan chartered accountant (CGNC chart of accounts).
You get one page of a balance sheet (bilan). Read the table and answer in JSON.

Copy the table exactly, row by row, from top to bottom. Do not compute, sum or skip anything.
- One entry per printed row, even if the row has no amounts. Total rows too.
- label: the row text exactly as printed (in French).
- values: the 4 amount columns, left to right: Brut, Amortissements et provisions, Net (current year), Net (previous year).
- Write amounts as numbers: "1 234 567,89" becomes 1234567.89. Keep the minus sign.
- Empty cell: null. Never invent an amount.
- Answer with the JSON only.

Example:
{"lignes": [{"label": "Frais préliminaires", "values": [2700.0, 2160.0, 540.0, 1080.0]}, {"label": "Primes de remboursement des obligations", "values": [null, null, null, null]}]}


In [7]:
import ollama

def ask(system, user, image, schema):
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": user, "images": [image]}],
        format=schema,
        options={"temperature": 0, "num_ctx": 16384, "num_predict": 8192},
    )
    return response.message.content

def norm(s):
    s = unicodedata.normalize("NFKD", s or "").encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[^a-z ]", " ", s)
    return " ".join(w for w in s.split() if len(w) > 1 or w == "i")

def similarity(label, code):
    a, b = norm(label), norm(LABELS[code][2])
    ratio = SequenceMatcher(None, a, b).ratio()
    return max(ratio, 0.9) if b and set(b.split()) <= set(a.split()) else ratio

def to_codes(rows, side):
    codes = CODES[side]
    sim = [[similarity(r["label"], c) for c in codes] for r in rows]
    best = [[0.0] * (len(codes) + 1) for _ in range(len(rows) + 1)]
    for i in range(1, len(rows) + 1):
        for j in range(1, len(codes) + 1):
            match = best[i - 1][j - 1] + sim[i - 1][j - 1] if sim[i - 1][j - 1] >= 0.45 else -1
            best[i][j] = max(best[i - 1][j], best[i][j - 1], match)
    found, i, j = {}, len(rows), len(codes)
    while i > 0 and j > 0:
        if sim[i - 1][j - 1] >= 0.45 and best[i][j] == best[i - 1][j - 1] + sim[i - 1][j - 1]:
            found[codes[j - 1]] = rows[i - 1]["values"]
            i, j = i - 1, j - 1
        elif best[i][j] == best[i - 1][j]:
            i -= 1
        else:
            j -= 1
    return found

def extract_page(setup, bilan, page, side):
    parts = images(setup, bilan, page)
    raw, rows, codes, ok = [], [], {}, 0
    for i, image in enumerate(parts):
        user = f"Page: BILAN {side}" + (f", part {i + 1} of {len(parts)}" if len(parts) > 1 else "")
        schema = codes_schema(side) if "codes" in setup else rows_schema(side)
        answer = ask(system_prompt(setup, side), user, image, schema)
        raw.append(answer)
        try:
            data = json.loads(answer)
        except Exception:
            continue
        ok += 1
        if "codes" in setup:
            codes.update({c: v for c, v in data.items() if c in LABELS})
        else:
            rows += [{"label": r["label"], "values": dict(zip(COLUMNS[side], r["values"]))} for r in data["lignes"]]
    if rows:
        codes = to_codes(rows, side)
    return codes, raw, ok == len(parts)

In [8]:
def value(found, code, col):
    v = (found.get(code) or {}).get(col) or 0.0
    if not v and code in ("ATR", "PTR"):
        v = (found.get(code[0] + "T3") or {}).get(col) or 0.0
    return v

def checks(found):
    results = []
    for side in COLUMNS:
        lines = [(c, LABELS[c][1], LABELS[c][3]) for c in CODES[side]]
        for col in ["n", "n1"]:
            for sec in {s for _, s, _ in lines} - {"TOTAL"}:
                sub = [c for c, s, k in lines if s == sec and k == "subtotal"]
                det = [c for c, s, k in lines if s == sec and k == "line"]
                if sub and det and (value(found, sub[0], col) or any(value(found, c, col) for c in det)):
                    results.append(abs(value(found, sub[0], col) - sum(SIGN.get(c, 1) * value(found, c, col) for c in det)) <= 1)
    return results

def accuracy(found, gold, nonzero_only):
    right = total = 0
    for code, cols in gold.items():
        for col in ["n", "n1"]:
            g = cols.get(col) or 0.0
            if nonzero_only and not g:
                continue
            total += 1
            right += abs(((found.get(code) or {}).get(col) or 0.0) - g) <= 0.01
    return round(right / total, 3)

def score(found, bilan, pages_ok, seconds):
    results = checks(found)
    metrics = {
        "json_valid": f"{pages_ok}/{len(PAGES[bilan])}",
        "codes_found": sum(1 for c in LABELS if c in found),
        "sums_ok": int(sum(results)),
        "sums_total": len(results),
        "balanced": all(value(found, "ATG", c) and abs(value(found, "ATG", c) - value(found, "PTG", c)) <= 1 for c in ["n", "n1"]),
        "seconds": round(seconds, 1),
    }
    if bilan in GOLD:
        metrics["acc_nonzero"] = accuracy(found, GOLD[bilan], True)
        metrics["acc_all"] = accuracy(found, GOLD[bilan], False)
    return metrics

def run(setup, bilan):
    config = {"model": MODEL, "setup": setup, "bilan": bilan,
              "prompt": hashlib.md5((system_prompt(setup, "ACTIF") + system_prompt(setup, "PASSIF")).encode()).hexdigest()[:8]}
    path = RESULTS / f"{setup}__{bilan}.json"
    if path.exists():
        old = json.loads(path.read_text(encoding="utf-8"))
        if json.dumps(old["config"], sort_keys=True) == json.dumps(config, sort_keys=True):
            print("already done:", old["metrics"])
            return old["metrics"]

    found, raw, pages_ok = {}, [], 0
    start = time.time()
    for page, side in enumerate(PAGES[bilan]):
        codes, answers, ok = extract_page(setup, bilan, page, side)
        raw += answers
        pages_ok += ok
        for code, values in codes.items():
            if sum(1 for v in values.values() if v) >= sum(1 for v in (found.get(code) or {}).values() if v):
                found[code] = values
    metrics = score(found, bilan, pages_ok, time.time() - start)

    path.write_text(json.dumps({"config": config, "date": datetime.datetime.now().isoformat(timespec="seconds"),
                                "metrics": metrics, "values": found, "raw": raw}, ensure_ascii=False, indent=1), encoding="utf-8")
    print(metrics)
    return metrics

def table():
    rows = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(RESULTS.glob("*.json"))]
    return pd.DataFrame([{"setup": r["config"]["setup"], "bilan": r["config"]["bilan"], **r["metrics"]} for r in rows])

In [9]:
run("codes", "6")

{'json_valid': '2/2', 'codes_found': 93, 'sums_ok': 8, 'sums_total': 20, 'balanced': True, 'seconds': 247.7, 'acc_nonzero': 0.821, 'acc_all': 0.817}


{'json_valid': '2/2',
 'codes_found': 93,
 'sums_ok': 8,
 'sums_total': 20,
 'balanced': True,
 'seconds': 247.7,
 'acc_nonzero': 0.821,
 'acc_all': 0.817}

In [10]:
run("transcribe", "6")

{'json_valid': '2/2', 'codes_found': 93, 'sums_ok': 9, 'sums_total': 20, 'balanced': True, 'seconds': 174.7, 'acc_nonzero': 0.747, 'acc_all': 0.839}


{'json_valid': '2/2',
 'codes_found': 93,
 'sums_ok': 9,
 'sums_total': 20,
 'balanced': True,
 'seconds': 174.7,
 'acc_nonzero': 0.747,
 'acc_all': 0.839}

In [11]:
run("transcribe+strips", "6")

{'json_valid': '2/2', 'codes_found': 93, 'sums_ok': 6, 'sums_total': 20, 'balanced': True, 'seconds': 196.6, 'acc_nonzero': 0.842, 'acc_all': 0.876}


{'json_valid': '2/2',
 'codes_found': 93,
 'sums_ok': 6,
 'sums_total': 20,
 'balanced': True,
 'seconds': 196.6,
 'acc_nonzero': 0.842,
 'acc_all': 0.876}

In [13]:
display(table())
BEST = table().sort_values("acc_nonzero", ascending=False).setup.iloc[0]
print("BEST =", BEST)

,setup,bilan,json_valid,codes_found,sums_ok,sums_total,balanced,seconds,acc_nonzero,acc_all
0,codes,6,2/2,93,8,20,True,247.7,0.821,0.817
1,transcribe+strips,6,2/2,93,6,20,True,196.6,0.842,0.876
2,transcribe,6,2/2,93,9,20,True,174.7,0.747,0.839


BEST = transcribe+strips


In [14]:
run(BEST, "3")

{'json_valid': '2/2', 'codes_found': 93, 'sums_ok': 14, 'sums_total': 14, 'balanced': True, 'seconds': 215.0, 'acc_nonzero': 1.0, 'acc_all': 1.0}


{'json_valid': '2/2',
 'codes_found': 93,
 'sums_ok': 14,
 'sums_total': 14,
 'balanced': True,
 'seconds': 215.0,
 'acc_nonzero': 1.0,
 'acc_all': 1.0}

In [15]:
run(BEST, "8")

{'json_valid': '2/3', 'codes_found': 75, 'sums_ok': 12, 'sums_total': 14, 'balanced': False, 'seconds': 561.0, 'acc_nonzero': 0.803, 'acc_all': 0.925}


{'json_valid': '2/3',
 'codes_found': 75,
 'sums_ok': 12,
 'sums_total': 14,
 'balanced': False,
 'seconds': 561.0,
 'acc_nonzero': 0.803,
 'acc_all': 0.925}

In [16]:
run(BEST, "10")

{'json_valid': '3/3', 'codes_found': 92, 'sums_ok': 11, 'sums_total': 14, 'balanced': True, 'seconds': 208.6, 'acc_nonzero': 0.896, 'acc_all': 0.952}


{'json_valid': '3/3',
 'codes_found': 92,
 'sums_ok': 11,
 'sums_total': 14,
 'balanced': True,
 'seconds': 208.6,
 'acc_nonzero': 0.896,
 'acc_all': 0.952}

In [17]:
run(BEST, "2")

{'json_valid': '2/2', 'codes_found': 92, 'sums_ok': 10, 'sums_total': 10, 'balanced': True, 'seconds': 179.0}


{'json_valid': '2/2',
 'codes_found': 92,
 'sums_ok': 10,
 'sums_total': 10,
 'balanced': True,
 'seconds': 179.0}

In [18]:
run(BEST, "4")

{'json_valid': '2/2', 'codes_found': 59, 'sums_ok': 2, 'sums_total': 16, 'balanced': False, 'seconds': 122.5}


{'json_valid': '2/2',
 'codes_found': 59,
 'sums_ok': 2,
 'sums_total': 16,
 'balanced': False,
 'seconds': 122.5}

In [19]:
results = table()
display(results)
best = results[results.setup == BEST]
print("average acc_nonzero:", round(best.acc_nonzero.mean(), 3), "| sums:", best.sums_ok.sum(), "/", best.sums_total.sum(),
      "| balanced:", int(best.balanced.sum()), "/", len(best), "| seconds per bilan:", round(best.seconds.mean()))

subprocess.run(["ollama", "stop", MODEL])

,setup,bilan,json_valid,codes_found,sums_ok,sums_total,balanced,seconds,acc_nonzero,acc_all
0,codes,6,2/2,93,8,20,True,247.7,0.821,0.817
1,transcribe+strips,10,3/3,92,11,14,True,208.6,0.896,0.952
2,transcribe+strips,2,2/2,92,10,10,True,179.0,NaN,NaN
3,transcribe+strips,3,2/2,93,14,14,True,215.0,1.000,1.000
4,transcribe+strips,4,2/2,59,2,16,False,122.5,NaN,NaN
5,transcribe+strips,6,2/2,93,6,20,True,196.6,0.842,0.876
6,transcribe+strips,8,2/3,75,12,14,False,561.0,0.803,0.925
7,transcribe,6,2/2,93,9,20,True,174.7,0.747,0.839
